In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1TQsq5gkCzWn6NvvP4CdwOAWH4xx8iweB", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/05_00_intro.mp3"))

In [ ]:
#@title 🎧 Listen: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_01_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_20_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Why Monitor
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_02_why_monitor.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

# Production Monitoring for Context Quality (Capstone) — Vizuara

# Production Monitoring for Context Quality
## Capstone: Building a Live Dashboard for Context Engineering

Bring everything together: compression, RAGAS, human evaluation, and A/B testing into a production monitoring system with drift detection and alerting.

**What you will learn:**
- Track context quality metrics in production
- EMA-based drift detection
- Build alerting systems with configurable thresholds

## 1. Why Does This Matter?

Context quality degrades over time. Without monitoring, you will not know until users complain.

In [ ]:
#@title 🎧 Listen: Install Imports
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_03_install_imports.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
!pip install -q numpy matplotlib pandas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime, timedelta
from collections import Counter

In [ ]:
#@title 🎧 Listen: Ema Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_04_ema_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Exponential Moving Average

$$\text{EMA}_t = \alpha \cdot x_t + (1 - \alpha) \cdot \text{EMA}_{t-1}$$

**Worked example:** $\alpha=0.2$, initial=0.85, scores=[0.83, 0.81, 0.79, 0.82, 0.78]

Day 1: $0.2 \times 0.83 + 0.8 \times 0.85 = 0.846$
Day 5: EMA drops to 0.818

In [ ]:
#@title 🎧 Listen: Ema Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_05_ema_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def ema(values, alpha, initial=None):
    result = []
    e = initial if initial is not None else values[0]
    for x in values:
        e = alpha * x + (1 - alpha) * e
        result.append(e)
    return result

scores = [0.83, 0.81, 0.79, 0.82, 0.78]
smoothed = ema(scores, 0.2, initial=0.85)
for day, (raw, sm) in enumerate(zip(scores, smoothed), 1):
    print(f"  Day {day}: raw={raw:.2f}, EMA={sm:.3f}")

In [ ]:
#@title 🎧 Listen: Monitor System Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_06_monitor_system_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Monitoring System

In [ ]:
#@title 🎧 Listen: Monitor Class
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_07_monitor_class.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class ContextQualityMonitor:
    def __init__(self, config=None):
        self.config = config or {
            "relevance": {"alpha": 0.2, "alert_below": 0.75, "target": 0.85},
            "faithfulness": {"alpha": 0.2, "alert_below": 0.80, "target": 0.90},
            "token_usage": {"alpha": 0.3, "alert_above": 80000, "target": 50000},
            "latency_ms": {"alpha": 0.3, "alert_above": 3000, "target": 1500},
        }
        self.ema_vals = {}
        self.history = {m: [] for m in self.config}
        self.ema_history = {m: [] for m in self.config}
        self.alerts = []

    def record(self, ts, metrics):
        new_alerts = []
        for m, v in metrics.items():
            if m not in self.config: continue
            cfg = self.config[m]
            self.history[m].append({"ts": ts, "v": v})

            if m not in self.ema_vals:
                self.ema_vals[m] = v
            else:
                self.ema_vals[m] = cfg["alpha"] * v + (1-cfg["alpha"]) * self.ema_vals[m]
            self.ema_history[m].append({"ts": ts, "v": self.ema_vals[m]})

            alert = False
            if "alert_below" in cfg and self.ema_vals[m] < cfg["alert_below"]:
                alert = True
                msg = f"[ALERT] {m} EMA={self.ema_vals[m]:.3f} < {cfg['alert_below']}"
            elif "alert_above" in cfg and self.ema_vals[m] > cfg["alert_above"]:
                alert = True
                msg = f"[ALERT] {m} EMA={self.ema_vals[m]:.0f} > {cfg['alert_above']}"

            if alert:
                a = {"ts": ts, "metric": m, "ema": self.ema_vals[m], "msg": msg}
                new_alerts.append(a)
                self.alerts.append(a)
        return new_alerts

    def status(self):
        s = {}
        for m, cfg in self.config.items():
            e = self.ema_vals.get(m)
            if e is None: s[m] = "NO DATA"; continue
            if "alert_below" in cfg:
                s[m] = f"HEALTHY ({e:.3f})" if e >= cfg["target"] else f"WARNING ({e:.3f})" if e >= cfg["alert_below"] else f"CRITICAL ({e:.3f})"
            else:
                s[m] = f"HEALTHY ({e:.0f})" if e <= cfg["target"] else f"WARNING ({e:.0f})" if e <= cfg["alert_above"] else f"CRITICAL ({e:.0f})"
        return s

monitor = ContextQualityMonitor()
print("Config:", {m: c for m, c in monitor.config.items()})

In [ ]:
#@title 🎧 Listen: Simulate Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_08_simulate_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Simulate 30 Days of Production Data

In [ ]:
#@title 🎧 Listen: Simulate Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_09_simulate_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
np.random.seed(42)
start = datetime(2026, 2, 1)
all_alerts = []

for day in range(30):
    ts = start + timedelta(days=day)
    if day < 15:
        rel = np.random.normal(0.85, 0.03)
        faith = np.random.normal(0.90, 0.02)
        tok = np.random.normal(45000, 5000)
        lat = np.random.normal(1200, 200)
    elif day < 25:
        decay = (day-15) * 0.008
        rel = np.random.normal(0.85-decay, 0.04)
        faith = np.random.normal(0.90-decay*0.5, 0.03)
        tok = np.random.normal(45000+(day-15)*3000, 5000)
        lat = np.random.normal(1200+(day-15)*100, 300)
    else:
        rel = np.random.normal(0.72, 0.05)
        faith = np.random.normal(0.82, 0.04)
        tok = np.random.normal(75000, 8000)
        lat = np.random.normal(2800, 500)

    alerts = monitor.record(ts, {
        "relevance": np.clip(rel, 0, 1),
        "faithfulness": np.clip(faith, 0, 1),
        "token_usage": max(0, tok),
        "latency_ms": max(0, lat)
    })
    all_alerts.extend(alerts)

print(f"30 days recorded. Alerts: {len(all_alerts)}")
for a in all_alerts[:5]:
    print(f"  {a['ts'].strftime('%m-%d')}: {a['msg']}")

In [ ]:
#@title 🎧 Listen: Current Status
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_10_current_status.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
print("\n=== Current Status ===")
for m, s in monitor.status().items():
    sym = "OK" if "HEALTHY" in s else "!!" if "WARNING" in s else "XX"
    print(f"  [{sym}] {m:<20} {s}")

In [ ]:
#@title 🎧 Listen: Todo Compound Alerts
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_11_todo_compound_alerts.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Your Turn

**TODO Exercise 1:** Implement compound alerts (fire when BOTH relevance AND faithfulness are below warning).

In [ ]:
def check_compound_alerts(monitor):
    """
    TODO: Check if relevance < 0.80 AND faithfulness < 0.85 simultaneously.
    Also check if token_usage > 70000 AND latency > 2500.
    Return list of alert strings.
    """
    # YOUR CODE HERE
    return []

In [ ]:
#@title 🎧 Listen: Todo Remediation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_12_todo_remediation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

**TODO Exercise 2:** Implement automatic remediation suggestions.

In [ ]:
def auto_remediate(metric):
    """
    TODO: Return remediation action for each metric:
    - relevance: "Re-index document corpus"
    - faithfulness: "Lower temperature, add grounding instruction"
    - token_usage: "Increase compression ratio"
    - latency_ms: "Enable caching"
    """
    # YOUR CODE HERE
    return f"No action for {metric}"

In [ ]:
#@title 🎧 Listen: Dashboard Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_13_dashboard_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Production Dashboard

In [ ]:
#@title 🎧 Listen: Dashboard Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_14_dashboard_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

plots = [
    ("relevance", "Context Relevance", "alert_below"),
    ("faithfulness", "Faithfulness", "alert_below"),
    ("token_usage", "Token Usage", "alert_above"),
    ("latency_ms", "Latency (ms)", "alert_above"),
]

for idx, (m, title, atype) in enumerate(plots):
    ax = axes[idx//2][idx%2]
    cfg = monitor.config[m]

    raw_d = [h["ts"] for h in monitor.history[m]]
    raw_v = [h["v"] for h in monitor.history[m]]
    ema_d = [h["ts"] for h in monitor.ema_history[m]]
    ema_v = [h["v"] for h in monitor.ema_history[m]]

    ax.scatter(raw_d, raw_v, color='gray', alpha=0.4, s=20, label='Daily')
    ax.plot(ema_d, ema_v, color='#4285F4', linewidth=2.5, label='EMA')

    thresh = cfg.get("alert_below", cfg.get("alert_above"))
    ax.axhline(y=thresh, color='red', linestyle='--', alpha=0.7, label='Threshold')
    ax.axhline(y=cfg["target"], color='green', linestyle='--', alpha=0.5, label='Target')

    for a in monitor.alerts:
        if a["metric"] == m:
            ax.axvline(x=a["ts"], color='red', alpha=0.15, linewidth=1)

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.2)

plt.suptitle('Context Quality Dashboard — 30 Days', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('production_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAlerts by metric:")
for m, c in Counter(a["metric"] for a in monitor.alerts).most_common():
    print(f"  {m}: {c}")

In [ ]:
#@title 🎧 Listen: Cycle Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_15_cycle_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. The Complete Cycle

In [ ]:
#@title 🎧 Listen: Cycle Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_16_cycle_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
print("=" * 55)
print("  THE OPTIMIZATION-EVALUATION CYCLE")
print("=" * 55)

print("\n1. DETECT")
status = monitor.status()
critical = [m for m, s in status.items() if "CRITICAL" in s]
print(f"   Critical: {critical}")

print("\n2. DIAGNOSE")
rel_h = [h["v"] for h in monitor.ema_history["relevance"]]
print(f"   Relevance: {rel_h[0]:.3f} -> {rel_h[-1]:.3f} (drop: {rel_h[0]-rel_h[-1]:.3f})")

print("\n3. OPTIMIZE")
print("   -> Re-index corpus, increase compression, add adaptive budgeting")

print("\n4. A/B TEST")
print("   -> Control: 72% relevance | Treatment: 83% relevance")
print("   -> z=3.1, p=0.002 => SIGNIFICANT")

print("\n5. DEPLOY & MONITOR")
print("   -> Deployed. Monitoring for 7 days.")
print("\n" + "=" * 55)

In [ ]:
#@title 🎧 Listen: Report Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_17_report_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Report Generator

In [ ]:
#@title 🎧 Listen: Report Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_18_report_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def monitoring_report(monitor):
    lines = ["=" * 55, "  CONTEXT QUALITY REPORT", f"  {datetime.now().strftime('%Y-%m-%d')}", "=" * 55]

    lines.append("\n--- STATUS ---")
    for m, s in monitor.status().items():
        lines.append(f"  {m:<20} {s}")

    lines.append("\n--- TRENDS ---")
    for m in monitor.config:
        if monitor.ema_history[m]:
            f = monitor.ema_history[m][0]["v"]
            l = monitor.ema_history[m][-1]["v"]
            lines.append(f"  {m:<20} {f:.3f} -> {l:.3f}")

    lines.append(f"\n--- ALERTS ({len(monitor.alerts)}) ---")
    for m, c in Counter(a["metric"] for a in monitor.alerts).most_common():
        lines.append(f"  {m}: {c}")

    lines.append("\n" + "=" * 55)
    return "\n".join(lines)

print(monitoring_report(monitor))

In [ ]:
#@title 🎧 Listen: Reflection Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_19_reflection_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Reflection

**Congratulations!** You have built a complete context quality monitoring system.

**Key takeaways from the entire series:**
- Context optimization without evaluation is guesswork
- Evaluation without monitoring is a one-time snapshot
- The cycle — measure, optimize, test, deploy, monitor — makes context engineering a true engineering discipline